# ETL - Encuesta de snacks Mundial FIFA 2026 (Guatemala)
## Secciones 1, 2 y 3: Datos generales, Habitos de consumo y Preferencias de producto


---

### Arquitectura del proceso

```
CSV Original
    ↓
Carga con Pandas
    ↓
Exploracion y diagnostico
    ↓
Limpieza (nulos, duplicados, texto)
    ↓
Transformacion y enriquecimiento
    ↓
Construccion de dimensiones y tabla de hechos
    ↓
KPIs y reportes para directivos
    ↓
Exportacion CSV para Power BI
```

---
## Paso 1. Importacion de librerias

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

---
## Paso 2. Carga del CSV

In [2]:
df_raw = pd.read_csv(
    'encuesta_snacks_mundial_2026_guatemala_2500_respuestas.csv'
)

print('Dimensiones del dataset:', df_raw.shape)
df_raw.head(3)

Dimensiones del dataset: (2500, 26)


,EncuestaID,FechaEncuesta,HoraEncuesta,RangoEdad,Genero,Departamento,Municipio,Ocupacion,FrecuenciaConsumoSnacks,LugarCompraSnacks,ConQuienVePartidos,GastoSnacksPartido,SnacksSeleccionados,SaborPreferido,PresentacionPreferida,PrecioAdecuado,PagaMasEdicionMundial,PlaneaVerMundial2026,SeleccionApoya,CompraDisenoSeleccion,SeleccionInfluyeCompra,JugadoresInfluyentes,TipoPublicidadAtractiva,PromocionPreferida,CompraTarjetasColeccionables,CampaniaMasProbableCompra
0,1,2026-04-15,09:46:21,55 años o más,Femenino,Guatemala,Santa Catarina Pinula,Jubilado,Varias veces al mes,Aplicación de delivery,Amigos,Q25 - Q50,FutbolMix; Hot Stadium Peanuts; Caramel Goal P...,DULCE,Familiar,Q10 - Q15,Sí,Sí,Portugal,Probablemente sí,Portugal,Kylian Mbappé; Cristiano Ronaldo; Neymar Jr.,Una selección nacional,Sorteo de viajes,Sí,Edición Francia
1,2,2026-01-26,08:45:33,18 - 24 años,Masculino,Guatemala,Santa Catarina Pinula,Empresario,1 a 2 veces por semana,Supermercado,Solo,Q51 - Q100,ChocoKick Bites; CrunchMax BBQ,Caramelo,Familiar,Q21 - Q30,Sí,Sí,Portugal,Probablemente no,Suecia,Cristiano Ronaldo; Kylian Mbappé; Christian Pu...,Una selección nacional,Producto gratis,Sí,Edición Brasil
2,3,2026-04-22,22:23:58,35 - 44 años,Masculino,Alta Verapaz,Alta Verapaz,Estudiante,Varias veces al mes,Kiosco escolar/universitario,Familia,Q51 - Q100,ChocoKick Bites; Fan Chips Jalapeño; Victory P...,Picante Chapín,Familiar,Q16 - Q20,Sí,Sí,España,Probablemente sí,Portugal,Cristiano Ronaldo; Federico Valverde,Un jugador famoso,Descuento directo,No,Edición con tarjetas coleccionables


---
## Paso 3. Seleccion de columnas de las secciones 1, 2 y 3

Se extraen unicamente las columnas correspondientes a las secciones asignadas.

In [3]:
columnas_s1_s2_s3 = [
    'EncuestaID',
    'FechaEncuesta',
    'RangoEdad',
    'Genero',
    'Departamento',
    'Municipio',
    'Ocupacion',
    'FrecuenciaConsumoSnacks',
    'LugarCompraSnacks',
    'ConQuienVePartidos',
    'GastoSnacksPartido',
    'SnacksSeleccionados',
    'SaborPreferido',
    'PresentacionPreferida'
]

df = df_raw[columnas_s1_s2_s3].copy()

print('Dimensiones del subset:', df.shape)
df.head()

Dimensiones del subset: (2500, 14)


,EncuestaID,FechaEncuesta,RangoEdad,Genero,Departamento,Municipio,Ocupacion,FrecuenciaConsumoSnacks,LugarCompraSnacks,ConQuienVePartidos,GastoSnacksPartido,SnacksSeleccionados,SaborPreferido,PresentacionPreferida
0,1,2026-04-15,55 años o más,Femenino,Guatemala,Santa Catarina Pinula,Jubilado,Varias veces al mes,Aplicación de delivery,Amigos,Q25 - Q50,FutbolMix; Hot Stadium Peanuts; Caramel Goal P...,DULCE,Familiar
1,2,2026-01-26,18 - 24 años,Masculino,Guatemala,Santa Catarina Pinula,Empresario,1 a 2 veces por semana,Supermercado,Solo,Q51 - Q100,ChocoKick Bites; CrunchMax BBQ,Caramelo,Familiar
2,3,2026-04-22,35 - 44 años,Masculino,Alta Verapaz,Alta Verapaz,Estudiante,Varias veces al mes,Kiosco escolar/universitario,Familia,Q51 - Q100,ChocoKick Bites; Fan Chips Jalapeño; Victory P...,Picante Chapín,Familiar
3,4,2026-01-28,25 - 34 años,Femenino,Guatemala,San Miguel Petapa,Empleado privado,3 a 5 veces por semana,Gasolinera,Solo,Q25 - Q50,Fan Chips Jalapeño; CrunchMax BBQ,Limón,Familiar
4,5,2026-03-04,25 - 34 años,Femenino,Guatemala,San Miguel Petapa,Estudiante,Varias veces al mes,Tienda de barrio,Amigos,Q51 - Q100,Caramel Goal Popcorn; FutbolMix,Queso,Para compartir entre 2 personas


---
## Paso 4. Exploracion y diagnostico de calidad

In [4]:
# Estructura y tipos de datos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   EncuestaID               2500 non-null   int64
 1   FechaEncuesta            2500 non-null   str  
 2   RangoEdad                2500 non-null   str  
 3   Genero                   2500 non-null   str  
 4   Departamento             2500 non-null   str  
 5   Municipio                2438 non-null   str  
 6   Ocupacion                2463 non-null   str  
 7   FrecuenciaConsumoSnacks  2500 non-null   str  
 8   LugarCompraSnacks        2470 non-null   str  
 9   ConQuienVePartidos       2500 non-null   str  
 10  GastoSnacksPartido       2500 non-null   str  
 11  SnacksSeleccionados      2500 non-null   str  
 12  SaborPreferido           2475 non-null   str  
 13  PresentacionPreferida    2500 non-null   str  
dtypes: int64(1), str(13)
memory usage: 273.6 KB


In [5]:
# Conteo de valores nulos por columna
nulos = df.isnull().sum().reset_index()
nulos.columns = ['Columna', 'Nulos']
nulos['Porcentaje'] = (nulos['Nulos'] / len(df) * 100).round(2)
nulos[nulos['Nulos'] > 0]

,Columna,Nulos,Porcentaje
5,Municipio,62,2.48
6,Ocupacion,37,1.48
8,LugarCompraSnacks,30,1.20
12,SaborPreferido,25,1.00


In [6]:
# Registros duplicados
print('Registros duplicados:', df.duplicated().sum())
print('EncuestaID duplicados:', df['EncuestaID'].duplicated().sum())

Registros duplicados: 16
EncuestaID duplicados: 25


---
## Paso 5. Limpieza de datos

### 5.1 Correccion de valores nulos

In [7]:
df['Ocupacion'] = (
    df['Ocupacion']
    .fillna('No Indicado')
)

df['LugarCompraSnacks'] = (
    df['LugarCompraSnacks']
    .fillna('No Indicado')
)

df['SaborPreferido'] = (
    df['SaborPreferido']
    .fillna('No Indicado')
)

print('Nulos restantes:', df.isnull().sum().sum())

Nulos restantes: 62


### 5.2 Normalizacion de texto

Se eliminando espacios extra

In [8]:
columnas_texto = [
    'RangoEdad',
    'Genero',
    'Departamento',
    'Municipio',
    'Ocupacion',
    'FrecuenciaConsumoSnacks',
    'LugarCompraSnacks',
    'ConQuienVePartidos',
    'GastoSnacksPartido',
    'SaborPreferido',
    'PresentacionPreferida'
]

for col in columnas_texto:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.title()
    )

print('Valores unicos de SaborPreferido despues de la limpieza:')
print(sorted(df['SaborPreferido'].unique()))

Valores unicos de SaborPreferido despues de la limpieza:
['Barbacoa', 'Caramelo', 'Chocolate', 'Dulce', 'Jalapeño', 'Limón', 'No Indicado', 'Picante Chapín', 'Queso', 'Salado Tradicional']


In [9]:
# Normalizacion del campo SnacksSeleccionados:
# eliminar espacios alrededor de cada snack en la lista separada por ';'
df['SnacksSeleccionados'] = (
    df['SnacksSeleccionados']
    .astype(str)
    .apply(
        lambda x: '; '.join(
            [s.strip() for s in x.split(';')]
        )
    )
)

df['SnacksSeleccionados'].head()

0    FutbolMix; Hot Stadium Peanuts; Caramel Goal P...
1                       ChocoKick Bites; CrunchMax BBQ
2    ChocoKick Bites; Fan Chips Jalapeño; Victory P...
3                    Fan Chips Jalapeño; CrunchMax BBQ
4                      Caramel Goal Popcorn; FutbolMix
Name: SnacksSeleccionados, dtype: str

### 5.3 Conversion de tipos de datos

In [10]:
df['FechaEncuesta'] = pd.to_datetime(df['FechaEncuesta'])

# Formato de fecha
meses_es = {
    1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril',
    5: 'Mayo', 6: 'Junio', 7: 'Julio', 8: 'Agosto',
    9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'
}
df['FechaEncuesta'] = df['FechaEncuesta'].apply(
    lambda f: f'{f.day:02d} {meses_es[f.month]} {f.year}'
)

df['EncuestaID'] = df['EncuestaID'].astype(int)

print('Tipos de datos:')
print(df[['EncuestaID', 'FechaEncuesta']].dtypes)

Tipos de datos:
EncuestaID       int64
FechaEncuesta      str
dtype: object


### 5.4 Ordenes logicos para columnas ordinales

Se definen listas de referencia para presentar los reportes en el orden correcto.

In [11]:
orden_edad = [
    'Menos De 18 Años',
    '18 - 24 Años',
    '25 - 34 Años',
    '35 - 44 Años',
    '45 - 54 Años',
    '55 Años O Mas'
]

orden_gasto = [
    'Menos De Q25',
    'Q25 - Q50',
    'Q51 - Q100',
    'Q101 - Q150',
    'Más De Q150'
]

orden_frecuencia = [
    'Rara Vez',
    'Varias Veces Al Mes',
    '1 A 2 Veces Por Semana',
    '3 A 5 Veces Por Semana',
    'Todos Los Dias'
]

print('Valores unicos de GastoSnacksPartido:')
print(sorted(df['GastoSnacksPartido'].unique()))
print()
print('Valores unicos de RangoEdad:')
print(sorted(df['RangoEdad'].unique()))

Valores unicos de GastoSnacksPartido:
['Menos De Q25', 'Más De Q150', 'Q101 - Q150', 'Q25 - Q50', 'Q51 - Q100']

Valores unicos de RangoEdad:
['18 - 24 Años', '25 - 34 Años', '35 - 44 Años', '45 - 54 Años', '55 Años O Más', 'Menos De 18 Años']


---
## Paso 6. Transformacion y enriquecimiento

### 6.1 Dimension tiempo desde la fecha de encuesta

In [12]:
df[['FechaEncuesta']].head()

,FechaEncuesta
0,15 Abril 2026
1,26 Enero 2026
2,22 Abril 2026
3,28 Enero 2026
4,04 Marzo 2026


### 6.2 Gasto promedio estimado por rango

Se asigna el punto medio de cada rango para calcular promedios numericos.

In [13]:
# Verificar valor exacto despues de .title() para mapear correctamente
valores_gasto = df['GastoSnacksPartido'].unique().tolist()
print('Valores exactos de GastoSnacksPartido:', valores_gasto)

Valores exactos de GastoSnacksPartido: ['Q25 - Q50', 'Q51 - Q100', 'Menos De Q25', 'Q101 - Q150', 'Más De Q150']


In [14]:
# El mapa usa los valores exactos que retorna .title()
mapa_gasto = {
    'Menos De Q25': 12.5,
    'Q25 - Q50': 37.5,
    'Q51 - Q100': 75.5,
    'Q101 - Q150': 125.5
}

# El valor 'Mas De Q150' puede conservar la tilde tras .title()
# Se mapean ambas variantes para garantizar cobertura total
for v in valores_gasto:
    if v not in mapa_gasto:
        mapa_gasto[v] = 175.0

df['GastoPromedio'] = df['GastoSnacksPartido'].map(mapa_gasto)

print('Cobertura del mapa:', df['GastoPromedio'].isnull().sum(), 'valores sin mapear')
print(f'Gasto promedio estimado global: Q{df["GastoPromedio"].mean():,.2f}')

Cobertura del mapa: 0 valores sin mapear
Gasto promedio estimado global: Q65.21


---
## Paso 7. Construccion del Star Schema

### 7.1 dim_encuestado

In [15]:
dim_encuestado = df[[
    'EncuestaID',
    'RangoEdad',
    'Genero',
    'Departamento',
    'Municipio',
    'Ocupacion'
]].copy()

print('dim_encuestado:', dim_encuestado.shape)
dim_encuestado.head()

dim_encuestado: (2500, 6)


,EncuestaID,RangoEdad,Genero,Departamento,Municipio,Ocupacion
0,1,55 Años O Más,Femenino,Guatemala,Santa Catarina Pinula,Jubilado
1,2,18 - 24 Años,Masculino,Guatemala,Santa Catarina Pinula,Empresario
2,3,35 - 44 Años,Masculino,Alta Verapaz,Alta Verapaz,Estudiante
3,4,25 - 34 Años,Femenino,Guatemala,San Miguel Petapa,Empleado Privado
4,5,25 - 34 Años,Femenino,Guatemala,San Miguel Petapa,Estudiante


### 7.2 dim_producto

In [16]:
dim_producto = df[[
    'EncuestaID',
    'SaborPreferido',
    'PresentacionPreferida'
]].copy()

print('dim_producto:', dim_producto.shape)
dim_producto.head()

dim_producto: (2500, 3)


,EncuestaID,SaborPreferido,PresentacionPreferida
0,1,Dulce,Familiar
1,2,Caramelo,Familiar
2,3,Picante Chapín,Familiar
3,4,Limón,Familiar
4,5,Queso,Para Compartir Entre 2 Personas


### 7.3 dim_snacks_detalle

Se desanida el campo `SnacksSeleccionados` (valores separados por `;`) para obtener una fila por snack. Esta estructura permite analisis correctos de frecuencia de mencion.

In [17]:
snacks_expandido = df[['EncuestaID', 'SnacksSeleccionados']].copy()

snacks_expandido['Snack'] = (
    snacks_expandido['SnacksSeleccionados']
    .str.split('; ')
)

dim_snacks_detalle = (
    snacks_expandido
    .explode('Snack')
    [['EncuestaID', 'Snack']]
    .reset_index(drop=True)
)

dim_snacks_detalle['Snack'] = (
    dim_snacks_detalle['Snack']
    .str.strip()
)

print('dim_snacks_detalle:', dim_snacks_detalle.shape)
print('Snacks unicos:', dim_snacks_detalle['Snack'].nunique())
dim_snacks_detalle.head(8)

dim_snacks_detalle: (5007, 2)
Snacks unicos: 10


,EncuestaID,Snack
0,1,FutbolMix
1,1,Hot Stadium Peanuts
2,1,Caramel Goal Popcorn
3,2,ChocoKick Bites
4,2,CrunchMax BBQ
5,3,ChocoKick Bites
6,3,Fan Chips Jalapeño
7,3,Victory Pretzels


### 7.4 fact_encuesta

In [18]:
fact_encuesta = df[[
    'EncuestaID',
    'FechaEncuesta',
    'FrecuenciaConsumoSnacks',
    'LugarCompraSnacks',
    'ConQuienVePartidos',
    'GastoSnacksPartido',
    'GastoPromedio'
]].copy()

print('fact_encuesta:', fact_encuesta.shape)
fact_encuesta.head()

fact_encuesta: (2500, 7)


,EncuestaID,FechaEncuesta,FrecuenciaConsumoSnacks,LugarCompraSnacks,ConQuienVePartidos,GastoSnacksPartido,GastoPromedio
0,1,15 Abril 2026,Varias Veces Al Mes,Aplicación De Delivery,Amigos,Q25 - Q50,37.50
1,2,26 Enero 2026,1 A 2 Veces Por Semana,Supermercado,Solo,Q51 - Q100,75.50
2,3,22 Abril 2026,Varias Veces Al Mes,Kiosco Escolar/Universitario,Familia,Q51 - Q100,75.50
3,4,28 Enero 2026,3 A 5 Veces Por Semana,Gasolinera,Solo,Q25 - Q50,37.50
4,5,04 Marzo 2026,Varias Veces Al Mes,Tienda De Barrio,Amigos,Q51 - Q100,75.50


---
## Paso 8. KPIs ejecutivos

In [19]:
total_encuestados = len(df)
total_departamentos = df['Departamento'].nunique()
total_municipios = df['Municipio'].nunique()
gasto_promedio_global = df['GastoPromedio'].mean()
total_menciones_snacks = len(dim_snacks_detalle)

print('=== KPIs Ejecutivos - Secciones 1, 2 y 3 ===')
print(f'Total de encuestados:          {total_encuestados:,}')
print(f'Departamentos cubiertos:       {total_departamentos}')
print(f'Municipios cubiertos:          {total_municipios}')
print(f'Gasto promedio estimado:       Q{gasto_promedio_global:,.2f}')
print(f'Total de menciones de snacks:  {total_menciones_snacks:,}')

=== KPIs Ejecutivos - Secciones 1, 2 y 3 ===
Total de encuestados:          2,500
Departamentos cubiertos:       24
Municipios cubiertos:          39
Gasto promedio estimado:       Q65.21
Total de menciones de snacks:  5,007


---
## Paso 9. Reportes para directivos

### 9.1 Perfil del consumidor por departamento y ocupacion

In [20]:
perfil_depto_ocupacion = (
    df.groupby(['Departamento', 'Ocupacion'])
    .agg(
        Encuestados=('EncuestaID', 'count')
    )
    .reset_index()
    .sort_values(['Departamento', 'Encuestados'], ascending=[True, False])
)

perfil_depto_ocupacion['Porcentaje'] = (
    perfil_depto_ocupacion['Encuestados']
    / total_encuestados * 100
).round(2)

print('Perfil del consumidor por departamento y ocupacion:')
perfil_depto_ocupacion.head(20)

Perfil del consumidor por departamento y ocupacion:


,Departamento,Ocupacion,Encuestados,Porcentaje
2,Alta Verapaz,Empleado Privado,15,0.60
5,Alta Verapaz,Estudiante,8,0.32
7,Alta Verapaz,Profesional Independiente,5,0.20
0,Alta Verapaz,Ama De Casa,4,0.16
4,Alta Verapaz,Empresario,4,0.16
1,Alta Verapaz,Comerciante,3,0.12
3,Alta Verapaz,Empleado Público,3,0.12
6,Alta Verapaz,Otro,2,0.08
9,Baja Verapaz,Comerciante,11,0.44
10,Baja Verapaz,Empleado Privado,7,0.28


### 9.2 Genero: porcentaje global, por edad agrupando por ocupacion

In [21]:
# Porcentaje global
genero_global = (
    df['Genero']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .reset_index()
)
genero_global.columns = ['Genero', 'Porcentaje']

print('Genero - Porcentaje global:')
print(genero_global.to_string(index=False))

Genero - Porcentaje global:
               Genero  Porcentaje
            Masculino       48.60
             Femenino       48.44
Prefiero No Indicarlo        2.96


In [22]:
# Por edad agrupando por ocupacion
genero_edad_ocupacion = (
    df.groupby(['RangoEdad', 'Ocupacion', 'Genero'])
    .agg(Encuestados=('EncuestaID', 'count'))
    .reset_index()
    .sort_values(['RangoEdad', 'Ocupacion', 'Encuestados'], ascending=[True, True, False])
)

genero_edad_ocupacion['Porcentaje'] = (
    genero_edad_ocupacion['Encuestados']
    / total_encuestados * 100
).round(2)

print('Genero por rango de edad y ocupacion:')
genero_edad_ocupacion.head(20)

Genero por rango de edad y ocupacion:


,RangoEdad,Ocupacion,Genero,Encuestados,Porcentaje
0,18 - 24 Años,Ama De Casa,Femenino,21,0.84
1,18 - 24 Años,Ama De Casa,Masculino,20,0.80
3,18 - 24 Años,Comerciante,Masculino,38,1.52
2,18 - 24 Años,Comerciante,Femenino,33,1.32
4,18 - 24 Años,Empleado Privado,Femenino,105,4.20
5,18 - 24 Años,Empleado Privado,Masculino,83,3.32
6,18 - 24 Años,Empleado Privado,Prefiero No Indicarlo,5,0.20
8,18 - 24 Años,Empleado Público,Masculino,25,1.00
7,18 - 24 Años,Empleado Público,Femenino,20,0.80
9,18 - 24 Años,Empresario,Femenino,22,0.88


### 9.3 Frecuencia de consumo: ranking con porcentajes, por edad agrupando por ocupacion

In [23]:
# Ranking global
frecuencia_rank = (
    df['FrecuenciaConsumoSnacks']
    .value_counts()
    .reset_index()
)
frecuencia_rank.columns = ['FrecuenciaConsumoSnacks', 'Encuestados']
frecuencia_rank['Porcentaje'] = (
    frecuencia_rank['Encuestados'] / total_encuestados * 100
).round(2)
frecuencia_rank.insert(0, 'Posicion', range(1, len(frecuencia_rank) + 1))

print('Ranking de frecuencia de consumo (global):')
print(frecuencia_rank.to_string(index=False))

Ranking de frecuencia de consumo (global):
 Posicion FrecuenciaConsumoSnacks  Encuestados  Porcentaje
        1  1 A 2 Veces Por Semana          698       27.92
        2  3 A 5 Veces Por Semana          674       26.96
        3     Varias Veces Al Mes          493       19.72
        4          Todos Los Días          466       18.64
        5                Rara Vez          169        6.76


In [24]:
# Por edad agrupando por ocupacion
frecuencia_edad_ocupacion = (
    df.groupby(['RangoEdad', 'Ocupacion', 'FrecuenciaConsumoSnacks'])
    .agg(Encuestados=('EncuestaID', 'count'))
    .reset_index()
    .sort_values(['RangoEdad', 'Ocupacion', 'Encuestados'], ascending=[True, True, False])
)

frecuencia_edad_ocupacion['Porcentaje'] = (
    frecuencia_edad_ocupacion['Encuestados'] / total_encuestados * 100
).round(2)

print('Frecuencia de consumo por rango de edad y ocupacion:')
frecuencia_edad_ocupacion.head(20)

Frecuencia de consumo por rango de edad y ocupacion:


,RangoEdad,Ocupacion,FrecuenciaConsumoSnacks,Encuestados,Porcentaje
1,18 - 24 Años,Ama De Casa,3 A 5 Veces Por Semana,13,0.52
0,18 - 24 Años,Ama De Casa,1 A 2 Veces Por Semana,12,0.48
3,18 - 24 Años,Ama De Casa,Todos Los Días,8,0.32
4,18 - 24 Años,Ama De Casa,Varias Veces Al Mes,7,0.28
2,18 - 24 Años,Ama De Casa,Rara Vez,1,0.04
6,18 - 24 Años,Comerciante,3 A 5 Veces Por Semana,24,0.96
5,18 - 24 Años,Comerciante,1 A 2 Veces Por Semana,22,0.88
8,18 - 24 Años,Comerciante,Todos Los Días,10,0.40
9,18 - 24 Años,Comerciante,Varias Veces Al Mes,10,0.40
7,18 - 24 Años,Comerciante,Rara Vez,5,0.20


### 9.4 Lugar de compra: ranking, top 3, por edad agrupando por ocupacion

In [25]:
# Ranking global
lugar_rank = (
    df['LugarCompraSnacks']
    .value_counts()
    .reset_index()
)
lugar_rank.columns = ['LugarCompraSnacks', 'Encuestados']
lugar_rank['Porcentaje'] = (
    lugar_rank['Encuestados'] / total_encuestados * 100
).round(2)
lugar_rank.insert(0, 'Posicion', range(1, len(lugar_rank) + 1))

print('Ranking de lugar de compra (global):')
print(lugar_rank.to_string(index=False))

Ranking de lugar de compra (global):
 Posicion            LugarCompraSnacks  Encuestados  Porcentaje
        1             Tienda De Barrio          776       31.04
        2                 Supermercado          770       30.80
        3                   Gasolinera          362       14.48
        4       Aplicación De Delivery          239        9.56
        5 Kiosco Escolar/Universitario          194        7.76
        6                         Otro          129        5.16
        7                  No Indicado           30        1.20


In [26]:
# Top 3 canales de compra
top3_lugar = lugar_rank.head(3)[['Posicion', 'LugarCompraSnacks', 'Porcentaje']]

print('Top 3 lugares de compra:')
print(top3_lugar.to_string(index=False))

Top 3 lugares de compra:
 Posicion LugarCompraSnacks  Porcentaje
        1  Tienda De Barrio       31.04
        2      Supermercado       30.80
        3        Gasolinera       14.48


In [27]:
# Por edad agrupando por ocupacion
lugar_edad_ocupacion = (
    df.groupby(['RangoEdad', 'Ocupacion', 'LugarCompraSnacks'])
    .agg(Encuestados=('EncuestaID', 'count'))
    .reset_index()
    .sort_values(['RangoEdad', 'Ocupacion', 'Encuestados'], ascending=[True, True, False])
)

lugar_edad_ocupacion['Porcentaje'] = (
    lugar_edad_ocupacion['Encuestados'] / total_encuestados * 100
).round(2)

print('Lugar de compra por rango de edad y ocupacion:')
lugar_edad_ocupacion.head(20)

Lugar de compra por rango de edad y ocupacion:


,RangoEdad,Ocupacion,LugarCompraSnacks,Encuestados,Porcentaje
4,18 - 24 Años,Ama De Casa,Supermercado,17,0.68
5,18 - 24 Años,Ama De Casa,Tienda De Barrio,11,0.44
1,18 - 24 Años,Ama De Casa,Gasolinera,6,0.24
2,18 - 24 Años,Ama De Casa,Kiosco Escolar/Universitario,4,0.16
0,18 - 24 Años,Ama De Casa,Aplicación De Delivery,2,0.08
3,18 - 24 Años,Ama De Casa,No Indicado,1,0.04
11,18 - 24 Años,Comerciante,Tienda De Barrio,25,1.00
10,18 - 24 Años,Comerciante,Supermercado,23,0.92
6,18 - 24 Años,Comerciante,Aplicación De Delivery,9,0.36
7,18 - 24 Años,Comerciante,Gasolinera,7,0.28


### 9.5 Con quien ve los partidos: ranking, por edad agrupando por ocupacion

In [28]:
# Ranking global
compania_rank = (
    df['ConQuienVePartidos']
    .value_counts()
    .reset_index()
)
compania_rank.columns = ['ConQuienVePartidos', 'Encuestados']
compania_rank['Porcentaje'] = (
    compania_rank['Encuestados'] / total_encuestados * 100
).round(2)
compania_rank.insert(0, 'Posicion', range(1, len(compania_rank) + 1))

print('Ranking - con quien ve los partidos (global):')
print(compania_rank.to_string(index=False))

Ranking - con quien ve los partidos (global):
 Posicion    ConQuienVePartidos  Encuestados  Porcentaje
        1               Familia         1030       41.20
        2                Amigos          910       36.40
        3 Compañeros De Trabajo          236        9.44
        4                  Solo          215        8.60
        5                  Otro          109        4.36


In [29]:
# Por edad agrupando por ocupacion
compania_edad_ocupacion = (
    df.groupby(['RangoEdad', 'Ocupacion', 'ConQuienVePartidos'])
    .agg(Encuestados=('EncuestaID', 'count'))
    .reset_index()
    .sort_values(['RangoEdad', 'Ocupacion', 'Encuestados'], ascending=[True, True, False])
)

compania_edad_ocupacion['Porcentaje'] = (
    compania_edad_ocupacion['Encuestados'] / total_encuestados * 100
).round(2)

print('Con quien ve los partidos por rango de edad y ocupacion:')
compania_edad_ocupacion.head(20)

Con quien ve los partidos por rango de edad y ocupacion:


,RangoEdad,Ocupacion,ConQuienVePartidos,Encuestados,Porcentaje
0,18 - 24 Años,Ama De Casa,Amigos,15,0.60
2,18 - 24 Años,Ama De Casa,Familia,14,0.56
4,18 - 24 Años,Ama De Casa,Solo,6,0.24
3,18 - 24 Años,Ama De Casa,Otro,4,0.16
1,18 - 24 Años,Ama De Casa,Compañeros De Trabajo,2,0.08
7,18 - 24 Años,Comerciante,Familia,32,1.28
5,18 - 24 Años,Comerciante,Amigos,20,0.80
9,18 - 24 Años,Comerciante,Solo,10,0.40
6,18 - 24 Años,Comerciante,Compañeros De Trabajo,5,0.20
8,18 - 24 Años,Comerciante,Otro,4,0.16


### 9.6 Presupuesto: ranking, promedio estimado, por edad agrupando por ocupacion

In [30]:
# Ranking global
gasto_rank = (
    df['GastoSnacksPartido']
    .value_counts()
    .reset_index()
)
gasto_rank.columns = ['GastoSnacksPartido', 'Encuestados']
gasto_rank['Porcentaje'] = (
    gasto_rank['Encuestados'] / total_encuestados * 100
).round(2)
gasto_rank.insert(0, 'Posicion', range(1, len(gasto_rank) + 1))

print('Ranking de presupuesto por partido (global):')
print(gasto_rank.to_string(index=False))
print()
print(f'Gasto promedio estimado global: Q{df["GastoPromedio"].mean():,.2f}')

Ranking de presupuesto por partido (global):
 Posicion GastoSnacksPartido  Encuestados  Porcentaje
        1          Q25 - Q50          867       34.68
        2         Q51 - Q100          736       29.44
        3       Menos De Q25          404       16.16
        4        Q101 - Q150          331       13.24
        5        Más De Q150          162        6.48

Gasto promedio estimado global: Q65.21


In [31]:
# Por edad agrupando por ocupacion con promedio estimado
gasto_edad_ocupacion = (
    df.groupby(['RangoEdad', 'Ocupacion', 'GastoSnacksPartido'])
    .agg(
        Encuestados=('EncuestaID', 'count'),
        GastoPromedioEstimado=('GastoPromedio', 'mean')
    )
    .reset_index()
    .sort_values(['RangoEdad', 'Ocupacion', 'Encuestados'], ascending=[True, True, False])
)

gasto_edad_ocupacion['Porcentaje'] = (
    gasto_edad_ocupacion['Encuestados'] / total_encuestados * 100
).round(2)

gasto_edad_ocupacion['GastoPromedioEstimado'] = (
    gasto_edad_ocupacion['GastoPromedioEstimado']
    .round(2)
)

print('Presupuesto por rango de edad y ocupacion:')
gasto_edad_ocupacion.head(20)

Presupuesto por rango de edad y ocupacion:


,RangoEdad,Ocupacion,GastoSnacksPartido,Encuestados,GastoPromedioEstimado,Porcentaje
3,18 - 24 Años,Ama De Casa,Q25 - Q50,19,37.50,0.76
4,18 - 24 Años,Ama De Casa,Q51 - Q100,9,75.50,0.36
0,18 - 24 Años,Ama De Casa,Menos De Q25,6,12.50,0.24
2,18 - 24 Años,Ama De Casa,Q101 - Q150,6,125.50,0.24
1,18 - 24 Años,Ama De Casa,Más De Q150,1,175.00,0.04
8,18 - 24 Años,Comerciante,Q25 - Q50,25,37.50,1.00
9,18 - 24 Años,Comerciante,Q51 - Q100,18,75.50,0.72
5,18 - 24 Años,Comerciante,Menos De Q25,10,12.50,0.40
6,18 - 24 Años,Comerciante,Más De Q150,9,175.00,0.36
7,18 - 24 Años,Comerciante,Q101 - Q150,9,125.50,0.36


### 9.7 Snacks favoritos: top 10, top 3, por edad agrupando por ocupacion

El analisis se realiza sobre `dim_snacks_detalle` para contar menciones individuales de forma correcta.

In [32]:
# Ranking global - el porcentaje indica penetracion sobre el total de encuestados
snack_rank = (
    dim_snacks_detalle['Snack']
    .value_counts()
    .reset_index()
)
snack_rank.columns = ['Snack', 'Menciones']
snack_rank['PorcentajeDeEncuestados'] = (
    snack_rank['Menciones'] / total_encuestados * 100
).round(2)
snack_rank.insert(0, 'Posicion', range(1, len(snack_rank) + 1))

print('Ranking de snacks favoritos (top 10):')
print(snack_rank.head(10).to_string(index=False))

Ranking de snacks favoritos (top 10):
 Posicion                     Snack  Menciones  PorcentajeDeEncuestados
        1 Mundialitos Queso Picante        692                    27.68
        2             CrunchMax BBQ        679                    27.16
        3        Gol Nachos Supreme        643                    25.72
        4        Fan Chips Jalapeño        580                    23.20
        5                 FutbolMix        574                    22.96
        6      Caramel Goal Popcorn        453                    18.12
        7           ChocoKick Bites        438                    17.52
        8       Hot Stadium Peanuts        420                    16.80
        9     Energy Crunch Protein        285                    11.40
       10          Victory Pretzels        243                     9.72


In [33]:
# Top 3
top3_snacks = snack_rank.head(3)[['Posicion', 'Snack', 'PorcentajeDeEncuestados']]

print('Top 3 snacks favoritos:')
print(top3_snacks.to_string(index=False))

Top 3 snacks favoritos:
 Posicion                     Snack  PorcentajeDeEncuestados
        1 Mundialitos Queso Picante                    27.68
        2             CrunchMax BBQ                    27.16
        3        Gol Nachos Supreme                    25.72


In [34]:
# Por edad agrupando por ocupacion:
# se une dim_snacks_detalle con dim_encuestado
snack_con_perfil = pd.merge(
    dim_snacks_detalle,
    dim_encuestado[['EncuestaID', 'RangoEdad', 'Ocupacion']],
    on='EncuestaID',
    how='left'
)

snack_edad_ocupacion = (
    snack_con_perfil
    .groupby(['RangoEdad', 'Ocupacion', 'Snack'])
    .agg(Menciones=('EncuestaID', 'count'))
    .reset_index()
    .sort_values(['RangoEdad', 'Ocupacion', 'Menciones'], ascending=[True, True, False])
)

snack_edad_ocupacion['PorcentajeDeEncuestados'] = (
    snack_edad_ocupacion['Menciones'] / total_encuestados * 100
).round(2)

print('Snacks favoritos por rango de edad y ocupacion (primeras filas):')
snack_edad_ocupacion.head(20)

Snacks favoritos por rango de edad y ocupacion (primeras filas):


,RangoEdad,Ocupacion,Snack,Menciones,PorcentajeDeEncuestados
0,18 - 24 Años,Ama De Casa,Caramel Goal Popcorn,13,0.52
6,18 - 24 Años,Ama De Casa,Gol Nachos Supreme,13,0.52
5,18 - 24 Años,Ama De Casa,FutbolMix,12,0.48
1,18 - 24 Años,Ama De Casa,ChocoKick Bites,10,0.40
3,18 - 24 Años,Ama De Casa,Energy Crunch Protein,9,0.36
7,18 - 24 Años,Ama De Casa,Hot Stadium Peanuts,9,0.36
8,18 - 24 Años,Ama De Casa,Mundialitos Queso Picante,9,0.36
2,18 - 24 Años,Ama De Casa,CrunchMax BBQ,7,0.28
4,18 - 24 Años,Ama De Casa,Fan Chips Jalapeño,6,0.24
15,18 - 24 Años,Comerciante,Gol Nachos Supreme,21,0.84


### 9.8 Sabor preferido: ranking, por edad agrupando por ocupacion

In [35]:
# Ranking global
sabor_rank = (
    df['SaborPreferido']
    .value_counts()
    .reset_index()
)
sabor_rank.columns = ['SaborPreferido', 'Encuestados']
sabor_rank['Porcentaje'] = (
    sabor_rank['Encuestados'] / total_encuestados * 100
).round(2)
sabor_rank.insert(0, 'Posicion', range(1, len(sabor_rank) + 1))

print('Ranking de sabor preferido (global):')
print(sabor_rank.to_string(index=False))

Ranking de sabor preferido (global):
 Posicion     SaborPreferido  Encuestados  Porcentaje
        1              Queso          431       17.24
        2     Picante Chapín          419       16.76
        3           Barbacoa          378       15.12
        4           Jalapeño          309       12.36
        5              Limón          278       11.12
        6          Chocolate          210        8.40
        7 Salado Tradicional          195        7.80
        8              Dulce          142        5.68
        9           Caramelo          113        4.52
       10        No Indicado           25        1.00


In [36]:
# Por edad agrupando por ocupacion
sabor_edad_ocupacion = (
    df.groupby(['RangoEdad', 'Ocupacion', 'SaborPreferido'])
    .agg(Encuestados=('EncuestaID', 'count'))
    .reset_index()
    .sort_values(['RangoEdad', 'Ocupacion', 'Encuestados'], ascending=[True, True, False])
)

sabor_edad_ocupacion['Porcentaje'] = (
    sabor_edad_ocupacion['Encuestados'] / total_encuestados * 100
).round(2)

print('Sabor preferido por rango de edad y ocupacion:')
sabor_edad_ocupacion.head(20)

Sabor preferido por rango de edad y ocupacion:


,RangoEdad,Ocupacion,SaborPreferido,Encuestados,Porcentaje
0,18 - 24 Años,Ama De Casa,Barbacoa,8,0.32
4,18 - 24 Años,Ama De Casa,Jalapeño,8,0.32
7,18 - 24 Años,Ama De Casa,Queso,8,0.32
6,18 - 24 Años,Ama De Casa,Picante Chapín,6,0.24
2,18 - 24 Años,Ama De Casa,Chocolate,5,0.20
8,18 - 24 Años,Ama De Casa,Salado Tradicional,3,0.12
1,18 - 24 Años,Ama De Casa,Caramelo,1,0.04
3,18 - 24 Años,Ama De Casa,Dulce,1,0.04
5,18 - 24 Años,Ama De Casa,Limón,1,0.04
16,18 - 24 Años,Comerciante,Queso,13,0.52


### 9.9 Presentacion preferida: ranking, porcentaje global, por edad agrupando por ocupacion

In [37]:
# Ranking global con porcentaje
presentacion_rank = (
    df['PresentacionPreferida']
    .value_counts()
    .reset_index()
)
presentacion_rank.columns = ['PresentacionPreferida', 'Encuestados']
presentacion_rank['Porcentaje'] = (
    presentacion_rank['Encuestados'] / total_encuestados * 100
).round(2)
presentacion_rank.insert(0, 'Posicion', range(1, len(presentacion_rank) + 1))

print('Ranking de presentacion preferida (global):')
print(presentacion_rank.to_string(index=False))

Ranking de presentacion preferida (global):
 Posicion           PresentacionPreferida  Encuestados  Porcentaje
        1                        Familiar          857       34.28
        2 Para Compartir Entre 2 Personas          639       25.56
        3                      Individual          577       23.08
        4         Tamaño Fiesta O Reunión          427       17.08


In [38]:
# Por edad agrupando por ocupacion
presentacion_edad_ocupacion = (
    df.groupby(['RangoEdad', 'Ocupacion', 'PresentacionPreferida'])
    .agg(Encuestados=('EncuestaID', 'count'))
    .reset_index()
    .sort_values(['RangoEdad', 'Ocupacion', 'Encuestados'], ascending=[True, True, False])
)

presentacion_edad_ocupacion['Porcentaje'] = (
    presentacion_edad_ocupacion['Encuestados'] / total_encuestados * 100
).round(2)

print('Presentacion preferida por rango de edad y ocupacion:')
presentacion_edad_ocupacion.head(20)

Presentacion preferida por rango de edad y ocupacion:


,RangoEdad,Ocupacion,PresentacionPreferida,Encuestados,Porcentaje
2,18 - 24 Años,Ama De Casa,Para Compartir Entre 2 Personas,18,0.72
1,18 - 24 Años,Ama De Casa,Individual,11,0.44
0,18 - 24 Años,Ama De Casa,Familiar,9,0.36
3,18 - 24 Años,Ama De Casa,Tamaño Fiesta O Reunión,3,0.12
4,18 - 24 Años,Comerciante,Familiar,26,1.04
5,18 - 24 Años,Comerciante,Individual,16,0.64
7,18 - 24 Años,Comerciante,Tamaño Fiesta O Reunión,15,0.60
6,18 - 24 Años,Comerciante,Para Compartir Entre 2 Personas,14,0.56
8,18 - 24 Años,Empleado Privado,Familiar,55,2.20
9,18 - 24 Años,Empleado Privado,Individual,54,2.16


---
## Paso 10. Tabla dinamica ejecutiva - Snacks por departamento

Menciones del top 3 de snacks por departamento para orientar decisiones de distribucion.

In [39]:
top3_lista = snack_rank.head(3)['Snack'].tolist()
print('Top 3 snacks seleccionados:', top3_lista)

snack_depto = pd.merge(
    dim_snacks_detalle,
    dim_encuestado[['EncuestaID', 'Departamento']],
    on='EncuestaID',
    how='left'
)

snack_depto_top3 = snack_depto[
    snack_depto['Snack'].isin(top3_lista)
]

pivot_snack_depto = pd.pivot_table(
    snack_depto_top3,
    values='EncuestaID',
    index='Departamento',
    columns='Snack',
    aggfunc='count',
    fill_value=0
)

print('Menciones del top 3 de snacks por departamento:')
pivot_snack_depto

Top 3 snacks seleccionados: ['Mundialitos Queso Picante', 'CrunchMax BBQ', 'Gol Nachos Supreme']
Menciones del top 3 de snacks por departamento:


Snack,CrunchMax BBQ,Gol Nachos Supreme,Mundialitos Queso Picante
Departamento,,,
Alta Verapaz,10,7,7
Baja Verapaz,14,7,14
Chimaltenango,47,45,39
Chiquimula,14,12,13
El Progreso,17,13,15
Escuintla,47,47,60
Guate,3,1,4
Guatemala,289,300,305
Huehuetenango,10,13,17


---
## Paso 11. Exportacion para Power BI

Se exportan las cuatro tablas del Star Schema y los reportes analiticos.

In [40]:
# Dimensiones y tabla de hechos
dim_encuestado.to_csv('dim_encuestado.csv', index=False)
dim_producto.to_csv('dim_producto.csv', index=False)
dim_snacks_detalle.to_csv('dim_snacks_detalle.csv', index=False)
fact_encuesta.to_csv('fact_encuesta.csv', index=False)

print('Dimensiones y tabla de hechos exportadas.')

Dimensiones y tabla de hechos exportadas.


In [41]:
# Reportes analiticos para directivos
snack_rank.to_csv('reporte_snacks_ranking.csv', index=False)
sabor_rank.to_csv('reporte_sabor_ranking.csv', index=False)
presentacion_rank.to_csv('reporte_presentacion_ranking.csv', index=False)
lugar_rank.to_csv('reporte_lugar_compra_ranking.csv', index=False)
frecuencia_rank.to_csv('reporte_frecuencia_consumo_ranking.csv', index=False)
gasto_rank.to_csv('reporte_presupuesto_ranking.csv', index=False)
perfil_depto_ocupacion.to_csv('reporte_perfil_depto_ocupacion.csv', index=False)
pivot_snack_depto.to_csv('reporte_pivot_snacks_departamento.csv')

print('Reportes analiticos exportados.')
print()
print('Archivos generados:')
archivos = [
    'dim_encuestado.csv',
    'dim_producto.csv',
    'dim_snacks_detalle.csv',
    'fact_encuesta.csv',
    'reporte_snacks_ranking.csv',
    'reporte_sabor_ranking.csv',
    'reporte_presentacion_ranking.csv',
    'reporte_lugar_compra_ranking.csv',
    'reporte_frecuencia_consumo_ranking.csv',
    'reporte_presupuesto_ranking.csv',
    'reporte_perfil_depto_ocupacion.csv',
    'reporte_pivot_snacks_departamento.csv'
]
for a in archivos:
    print(' -', a)

Reportes analiticos exportados.

Archivos generados:
 - dim_encuestado.csv
 - dim_producto.csv
 - dim_snacks_detalle.csv
 - fact_encuesta.csv
 - reporte_snacks_ranking.csv
 - reporte_sabor_ranking.csv
 - reporte_presentacion_ranking.csv
 - reporte_lugar_compra_ranking.csv
 - reporte_frecuencia_consumo_ranking.csv
 - reporte_presupuesto_ranking.csv
 - reporte_perfil_depto_ocupacion.csv
 - reporte_pivot_snacks_departamento.csv


---
## Resumen de archivos generados

| Archivo | Descripcion | Uso |
|---|---|---|
| `dim_encuestado.csv` | Perfil demografico por EncuestaID | Dimension en Power BI |
| `dim_producto.csv` | Sabor y presentacion por EncuestaID | Dimension en Power BI |
| `dim_snacks_detalle.csv` | Una fila por snack seleccionado | Dimension en Power BI |
| `fact_encuesta.csv` | Hechos de habitos y gasto | Tabla de hechos en Power BI |
| `reporte_snacks_ranking.csv` | Top 10 snacks con menciones | Reporte para directivos |
| `reporte_sabor_ranking.csv` | Ranking de sabores preferidos | Reporte para directivos |
| `reporte_presentacion_ranking.csv` | Ranking de presentaciones | Reporte para directivos |
| `reporte_lugar_compra_ranking.csv` | Ranking de canales de compra | Reporte para directivos |
| `reporte_frecuencia_consumo_ranking.csv` | Ranking de frecuencia | Reporte para directivos |
| `reporte_presupuesto_ranking.csv` | Ranking de rangos de gasto | Reporte para directivos |
| `reporte_perfil_depto_ocupacion.csv` | Perfil cruzado departamento-ocupacion | Decision de distribucion |
| `reporte_pivot_snacks_departamento.csv` | Top 3 snacks por departamento | Decision de distribucion |

Todas las tablas se conectan mediante `EncuestaID`, lo que permite al companero integrar las secciones 4, 5 y 6 sobre el mismo Star Schema.